`## Kan examples for multiplier

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim


# --------------------------
# 1. Univariate KAN edge function
#    phi(x) = a*x + b*x^2 + c
# --------------------------
class BasisEdge(nn.Module):
    def __init__(self):
        super().__init__()
        self.a = nn.Parameter(torch.randn(1) * 0.1)
        self.b = nn.Parameter(torch.randn(1) * 0.1)
        self.c = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        return self.a * x + self.b * x**2 + self.c


# --------------------------
# 2. Tiny KAN-style network
#
# Hidden nodes:
#   h1 = phi11(x1) + phi12(x2)
#   h2 = phi21(x1)
#   h3 = phi31(x2)
#
# Output:
#   y  = psi1(h1) + psi2(h2) + psi3(h3)
#
# All phi / psi are univariate basis expansions in x and x^2.
# Only summation is used between them.
# --------------------------
class TinyKANMultiplier(nn.Module):
    def __init__(self):
        super().__init__()

        # input -> hidden
        self.phi11 = BasisEdge()
        self.phi12 = BasisEdge()
        self.phi21 = BasisEdge()
        self.phi31 = BasisEdge()

        # hidden -> output
        self.psi1 = BasisEdge()
        self.psi2 = BasisEdge()
        self.psi3 = BasisEdge()

    def forward(self, x):
        x1 = x[:, 0:1]
        x2 = x[:, 1:2]

        h1 = self.phi11(x1) + self.phi12(x2)
        h2 = self.phi21(x1)
        h3 = self.phi31(x2)

        y = self.psi1(h1) + self.psi2(h2) + self.psi3(h3)
        return y


# --------------------------
# 3. Training data
# --------------------------
torch.manual_seed(0)

N = 2000
x = 2 * torch.rand(N, 2) - 1   # uniform in [-1, 1]^2
y = (x[:, 0] * x[:, 1]).unsqueeze(1)

# optional train/test split
perm = torch.randperm(N)
train_idx = perm[:1600]
test_idx = perm[1600:]

x_train, y_train = x[train_idx], y[train_idx]
x_test, y_test = x[test_idx], y[test_idx]


# --------------------------
# 4. Train
# --------------------------
model = TinyKANMultiplier()
optimizer = optim.Adam(model.parameters(), lr=1e-2)
loss_fn = nn.MSELoss()

for epoch in range(3000):
    pred = model(x_train)
    loss = loss_fn(pred, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 300 == 0:
        with torch.no_grad():
            test_loss = loss_fn(model(x_test), y_test).item()
        print(f"epoch={epoch:4d}  train_loss={loss.item():.6e}  test_loss={test_loss:.6e}")


# --------------------------
# 5. Inspect a few predictions
# --------------------------
with torch.no_grad():
    sample = torch.tensor([
        [0.2, 0.3],
        [0.5, -0.4],
        [-0.7, -0.8],
        [1.0, 0.25],
    ])
    pred = model(sample).squeeze()
    true = (sample[:, 0] * sample[:, 1])

    print("\nSample predictions:")
    for s, p, t in zip(sample, pred, true):
        print(f"x1={s[0]: .3f}, x2={s[1]: .3f}, pred={p.item(): .6f}, true={t.item(): .6f}")

In [ ]:
import torch
import torch.nn as nn

class ExactMultiplierKAN(nn.Module):
    def forward(self, x):
        x1 = x[:, 0:1]
        x2 = x[:, 1:2]
        return 0.5 * (x1 + x2) ** 2 - 0.5 * x1 ** 2 - 0.5 * x2 ** 2


model = ExactMultiplierKAN()

sample = torch.tensor([
    [0.2, 0.3],
    [0.5, -0.4],
    [-0.7, -0.8],
    [1.0, 0.25],
])

pred = model(sample).squeeze()
true = sample[:, 0] * sample[:, 1]

for s, p, t in zip(sample, pred, true):
    print(f"x1={s[0]: .3f}, x2={s[1]: .3f}, pred={p.item(): .6f}, true={t.item(): .6f}")